# Japanese StackExchange → Argilla Filtering Pipeline

Load, filter, and push the p1atdev/japanese-stackexchange dataset to your Argilla Space for curation.

**Workflow:**
1. Connect to your Argilla HF Space
2. Load dataset from HuggingFace (pinned revision)
3. Apply gross filtering (tags, keywords, quality)
4. Push filtered subset to Argilla for fine-grained curation

**Argilla version:** v2.x

## Setup

In [ ]:
# Install dependencies (Colab)
!pip install -q "argilla>=2.0" datasets

In [ ]:
import argilla as rg
from datasets import load_dataset
import re
from typing import Any

print(f"Argilla version: {rg.__version__}")

## Configuration

Update these values for your setup:

In [ ]:
# === ARGILLA CONNECTION ===
# Credentials loaded from Colab secrets (recommended) or environment variables
import os
try:
    from google.colab import userdata
    ARGILLA_API_URL = userdata.get("ARGILLA_API_URL")
    ARGILLA_API_KEY = userdata.get("ARGILLA_API_KEY")
    HF_TOKEN = userdata.get("HF_TOKEN")  # Required for HF Space auth
except (ImportError, ModuleNotFoundError):
    # Not in Colab — fall back to environment
    ARGILLA_API_URL = os.getenv("ARGILLA_API_URL")
    ARGILLA_API_KEY = os.getenv("ARGILLA_API_KEY")
    HF_TOKEN = os.getenv("HF_TOKEN")

if not ARGILLA_API_URL or not ARGILLA_API_KEY:
    raise ValueError(
        "Missing credentials!\n"
        "In Colab: Add ARGILLA_API_URL, ARGILLA_API_KEY, and HF_TOKEN to Colab Secrets (key icon in sidebar)\n"
        "Locally: export ARGILLA_API_URL, ARGILLA_API_KEY, and HF_TOKEN environment variables"
    )

if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — HF Space authentication may fail")

print(f"Argilla URL: {ARGILLA_API_URL}")

# === SOURCE DATASET ===
SOURCE_DATASET = "p1atdev/japanese-stackexchange"
SOURCE_SUBSET = "simple"  # 'simple' has flattened structure, easier to work with
SOURCE_REVISION = "4d65476f8b1d9a140a15e01a4c21e3a39d7b12d9"  # Pinned for reproducibility

# === OUTPUT ===
# Two workflows available:
#   - "jse-grammar-candidates": filtered subset (cells 16-26)
#   - "jse-full": entire dataset (cell 14)
ARGILLA_DATASET_NAME = "jse-grammar-candidates"
ARGILLA_DATASET_NAME_FULL = "jse-full"

## Filter Configuration

Define your gross filtering criteria:

In [ ]:
# === TAG FILTERS ===
# Include if ANY of these tags are present (empty list = no tag filter)
INCLUDE_TAGS = [
    "grammar",
    "meaning",
    "particles",
    "conjugation",
    "conjugations",
    "verbs",
    "adjectives",
    "sentence-structure",
    "translation",
    "て-form",
    "particle-は",
    "particle-を",
    "particle-で",
    "relative-clauses",
    "passive-voice",
    "conditionals",
]

# === KEYWORD FILTERS ===
# Regex patterns to match in question title or body (empty list = no keyword filter)
# Uses re.IGNORECASE
TITLE_PATTERNS = [
    r"difference between",
    r"how to use",
    r"when to use",
    r"meaning of",
    r"vs\.?",
    r"how do I say",
    r"how could I say",
]

BODY_PATTERNS = [
    r"て形|te-form|te form",
    r"ない形|nai-form|negative form",
    r"ば|たら|なら|と\s",  # conditionals
    r"ように|ために",  # purpose
    r"ことができる|られる",  # potential
    r"てしまう|ちゃう",  # completion/regret
    r"けど|が|でも",  # conjunctions
    r"は[^a-z]|が[^a-z]",  # wa/ga particles (not in English words)
]

# === QUALITY FILTERS ===
MIN_QUESTION_SCORE = 2  # Minimum upvotes on question
REQUIRE_ACCEPTED_ANSWER = True  # Must have accepted answer
MIN_ACCEPTED_ANSWER_SCORE = 2  # Minimum score on accepted answer (None = no filter)
MIN_POPULAR_ANSWER_SCORE = 2  # Minimum score on highest-voted answer

## Connect to Argilla

In [ ]:
# Initialise Argilla client (v2 API)
# HF_TOKEN header required for HF Space authentication
client = rg.Argilla(
    api_url=ARGILLA_API_URL,
    api_key=ARGILLA_API_KEY,
    headers={"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else None,
)

# Test connection
try:
    datasets = client.datasets.list()
    print(f"Connected! Found {len(datasets)} existing datasets:")
    for ds in datasets:
        print(f"  - {ds.name}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check your ARGILLA_API_URL, ARGILLA_API_KEY, and HF_TOKEN")

## Load Source Dataset

In [ ]:
print(f"Loading {SOURCE_DATASET} ({SOURCE_SUBSET}) @ {SOURCE_REVISION}...")

ds = load_dataset(
    SOURCE_DATASET,
    SOURCE_SUBSET,
    revision=SOURCE_REVISION,
    split="train"
)

print(f"Loaded {len(ds):,} records")
print(f"\nColumns: {ds.column_names}")
print(f"\nSample record:")
ds[0]

## Explore Tags (Optional)

See what tags are available in the dataset:

In [ ]:
from collections import Counter

# Count all tags
tag_counts = Counter()
for record in ds:
    tag_counts.update(record["tags"])

print(f"Found {len(tag_counts)} unique tags\n")
print("Top 30 tags:")
for tag, count in tag_counts.most_common(30):
    print(f"  {tag}: {count}")

In [ ]:
# === Quick Load: Push entire dataset to Argilla (batched) ===
# Skip filtering — curate everything in Argilla instead
# REQUIRES: Run cells 3-5 first (imports + config)

from datasets import load_dataset
from tqdm.auto import tqdm
import argilla as rg

BATCH_SIZE = 500

# Ensure client is initialised (idempotent) with HF_TOKEN for Space auth
client = rg.Argilla(
    api_url=ARGILLA_API_URL,
    api_key=ARGILLA_API_KEY,
    headers={"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else None,
)

# Load dataset using config variables
print(f"Loading {SOURCE_DATASET} ({SOURCE_SUBSET}) @ {SOURCE_REVISION}...")
ds = load_dataset(SOURCE_DATASET, SOURCE_SUBSET, revision=SOURCE_REVISION, split="train")
print(f"Loaded {len(ds):,} records")

# Define Argilla dataset settings
settings = rg.Settings(
    fields=[
        rg.TextField(name="title", title="Question Title"),
        rg.TextField(name="question_body", title="Question", use_markdown=True),
        rg.TextField(name="accepted_answer_body", title="Accepted Answer", use_markdown=True, required=False),
        rg.TextField(name="popular_answer_body", title="Popular Answer", use_markdown=True, required=False),
        rg.TextField(name="tags", title="Tags"),
    ],
    questions=[
        rg.LabelQuestion(
            name="grammar_relevance",
            title="Is this relevant for grammar point training?",
            labels=[
                "yes_high_quality",
                "yes_needs_editing",
                "no_off_topic",
                "no_too_simple",
                "no_too_complex",
            ],
        ),
        rg.MultiLabelQuestion(
            name="grammar_topics",
            title="What grammar topics does this cover?",
            labels=[
                "particles",
                "verb_conjugation",
                "adjectives",
                "conditionals",
                "honorifics",
                "sentence_structure",
                "set_phrases",
                "other",
            ],
            required=False,
        ),
        rg.TextQuestion(
            name="notes",
            title="Notes (optional)",
            required=False,
        ),
    ],
    metadata=[
        rg.IntegerMetadataProperty(name="question_score", title="Question Score"),
        rg.TermsMetadataProperty(name="original_tags", title="Original Tags"),
    ],
)

# Create dataset in Argilla
dataset = rg.Dataset(name=ARGILLA_DATASET_NAME_FULL, settings=settings)

try:
    dataset.create()
    print(f"Created dataset '{ARGILLA_DATASET_NAME_FULL}'")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Dataset '{ARGILLA_DATASET_NAME_FULL}' already exists, fetching...")
        dataset = client.datasets(name=ARGILLA_DATASET_NAME_FULL)
    else:
        raise e

# Upload in batches with progress bar
def make_record(r):
    return rg.Record(
        fields={
            "title": r["title"] or "",
            "question_body": r["question_body"] or "",
            "accepted_answer_body": r["accepted_answer_body"] or "(No accepted answer)",
            "popular_answer_body": r["popular_answer_body"] or "(No popular answer)",
            "tags": ", ".join(r["tags"]) if r["tags"] else "",
        },
        metadata={
            "question_score": r["question_score"] or 0,
            "original_tags": r["tags"] or [],
        },
        id=r["id"],
    )

print(f"\nUploading {len(ds):,} records in batches of {BATCH_SIZE}...")
for i in tqdm(range(0, len(ds), BATCH_SIZE), desc="Batches"):
    batch = ds.select(range(i, min(i + BATCH_SIZE, len(ds))))
    records = [make_record(r) for r in batch]
    dataset.records.log(records)

print(f"\n✓ Done! Logged {len(ds):,} records to '{ARGILLA_DATASET_NAME_FULL}'")

In [ ]:
# === Comprehensive Tag Analysis ===
# Run this to understand the tag distribution before filtering

from collections import Counter
import json

# Count all tags
tag_counts = Counter()
for record in ds:
    tag_counts.update(record["tags"])

total_tags = sum(tag_counts.values())
unique_tags = len(tag_counts)

print(f"📊 Tag Statistics")
print(f"{'─' * 40}")
print(f"Total tag occurrences: {total_tags:,}")
print(f"Unique tags: {unique_tags}")
print(f"Average tags per question: {total_tags / len(ds):.1f}")
print()

# Full tag list sorted by frequency
print(f"📋 All Tags by Frequency ({unique_tags} unique)")
print(f"{'─' * 40}")

# Create a structured output
all_tags = []
for i, (tag, count) in enumerate(tag_counts.most_common(), 1):
    pct = count / len(ds) * 100
    all_tags.append({"tag": tag, "count": count, "pct": pct})
    # Print with alignment
    print(f"{i:3}. {tag:40} {count:5,} ({pct:5.1f}%)")

print()

# Suggest grammar-related tags (heuristic matching)
grammar_keywords = [
    "grammar", "verb", "particle", "conjugat", "tense", "form",
    "adjective", "noun", "adverb", "sentence", "clause", "phrase",
    "passive", "causative", "potential", "honorific", "polite",
    "formal", "casual", "keigo", "expression", "suffix", "prefix",
]

print(f"🎯 Potentially Grammar-Related Tags")
print(f"{'─' * 40}")
grammar_tags = []
for tag, count in tag_counts.most_common():
    if any(kw in tag.lower() for kw in grammar_keywords):
        grammar_tags.append((tag, count))
        pct = count / len(ds) * 100
        print(f"  {tag:40} {count:5,} ({pct:5.1f}%)")

print()

# Export as Python list for easy copy-paste into config
print(f"📝 Copy-Paste for INCLUDE_TAGS (grammar-related):")
print(f"{'─' * 40}")
print("INCLUDE_TAGS = [")
for tag, _ in grammar_tags:
    print(f'    "{tag}",')
print("]")

print()

# Also show non-grammar tags for context
non_grammar = [(t, c) for t, c in tag_counts.most_common(50) 
               if not any(kw in t.lower() for kw in grammar_keywords)]
print(f"📌 Top Non-Grammar Tags (for exclusion or separate filtering)")
print(f"{'─' * 40}")
for tag, count in non_grammar[:20]:
    pct = count / len(ds) * 100
    print(f"  {tag:40} {count:5,} ({pct:5.1f}%)")

## Apply Filters

In [ ]:
def matches_any_pattern(text: str | None, patterns: list[str]) -> bool:
    """Check if text matches any of the regex patterns."""
    if not text or not patterns:
        return not patterns  # No patterns = pass
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def passes_filter(record: dict[str, Any]) -> bool:
    """Apply all filtering criteria to a record."""
    
    # Tag filter (any match)
    if INCLUDE_TAGS:
        if not any(tag in INCLUDE_TAGS for tag in record.get("tags", [])):
            return False
    
    # Keyword filters (any match in title OR body)
    title_match = matches_any_pattern(record.get("title"), TITLE_PATTERNS)
    body_match = matches_any_pattern(record.get("question_body"), BODY_PATTERNS)
    
    if TITLE_PATTERNS or BODY_PATTERNS:
        if not (title_match or body_match):
            return False
    
    # Quality: minimum question score
    if MIN_QUESTION_SCORE is not None:
        if (record.get("question_score") or 0) < MIN_QUESTION_SCORE:
            return False
    
    # Quality: require accepted answer
    if REQUIRE_ACCEPTED_ANSWER:
        if not record.get("accepted_answer_body"):
            return False
    
    # Quality: minimum accepted answer score
    if MIN_ACCEPTED_ANSWER_SCORE is not None:
        if (record.get("accepted_answer_score") or 0) < MIN_ACCEPTED_ANSWER_SCORE:
            return False
    
    # Quality: minimum popular answer score
    if MIN_POPULAR_ANSWER_SCORE is not None:
        if (record.get("popular_answer_score") or 0) < MIN_POPULAR_ANSWER_SCORE:
            return False
    
    return True


# Apply filter
print("Applying filters...")
filtered_ds = ds.filter(passes_filter)

print(f"\nFiltered: {len(ds):,} → {len(filtered_ds):,} records")
print(f"Retention: {len(filtered_ds) / len(ds) * 100:.1f}%")

## Preview Filtered Results

In [ ]:
# Show a few examples
print("Sample filtered records:\n")
for i, record in enumerate(filtered_ds.select(range(min(5, len(filtered_ds))))):
    print(f"--- Record {i+1} ---")
    print(f"Title: {record['title']}")
    print(f"Tags: {record['tags']}")
    print(f"Score: {record['question_score']}")
    print(f"Body preview: {record['question_body'][:200]}...\n")

## Create Argilla Dataset (v2 API)

Define the dataset schema with fields for viewing and questions for annotation.

In [ ]:
# Define dataset settings (v2 API)
settings = rg.Settings(
    fields=[
        rg.TextField(name="title", title="Question Title"),
        rg.TextField(name="question_body", title="Question", use_markdown=True),
        rg.TextField(name="accepted_answer_body", title="Accepted Answer", use_markdown=True, required=False),
        rg.TextField(name="popular_answer_body", title="Popular Answer", use_markdown=True, required=False),
        rg.TextField(name="tags", title="Tags"),
    ],
    questions=[
        rg.LabelQuestion(
            name="grammar_relevance",
            title="Is this relevant for grammar point training?",
            labels=[
                "yes_high_quality",
                "yes_needs_editing",
                "no_off_topic",
                "no_too_simple",
                "no_too_complex",
            ],
        ),
        rg.MultiLabelQuestion(
            name="grammar_topics",
            title="What grammar topics does this cover?",
            labels=[
                "particles",
                "verb_conjugation",
                "adjectives",
                "conditionals",
                "honorifics",
                "sentence_structure",
                "set_phrases",
                "other",
            ],
            required=False,
        ),
        rg.TextQuestion(
            name="notes",
            title="Notes (optional)",
            required=False,
        ),
    ],
    metadata=[
        rg.IntegerMetadataProperty(name="question_score", title="Question Score"),
        rg.TermsMetadataProperty(name="original_tags", title="Original Tags"),
    ],
)

print("Settings created")

In [ ]:
# Create the dataset
dataset = rg.Dataset(
    name=ARGILLA_DATASET_NAME,
    settings=settings,
)

# Push to Argilla (creates empty dataset with schema)
try:
    dataset.create()
    print(f"Created dataset '{ARGILLA_DATASET_NAME}'")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Dataset '{ARGILLA_DATASET_NAME}' already exists.")
        print("Run the deletion cell below if you want to replace it.")
    else:
        raise e

In [ ]:
# Optional: Delete existing dataset
# Uncomment to delete and recreate

# existing = client.datasets(name=ARGILLA_DATASET_NAME)
# if existing:
#     existing.delete()
#     print(f"Deleted '{ARGILLA_DATASET_NAME}'")
#     dataset.create()
#     print(f"Recreated '{ARGILLA_DATASET_NAME}'")

## Add Records

In [ ]:
# Get the dataset (in case it already existed)
dataset = client.datasets(name=ARGILLA_DATASET_NAME)

# Convert filtered HF dataset to Argilla records
records = []
for record in filtered_ds:
    records.append(
        rg.Record(
            fields={
                "title": record["title"] or "",
                "question_body": record["question_body"] or "",
                "accepted_answer_body": record["accepted_answer_body"] or "(No accepted answer)",
                "popular_answer_body": record["popular_answer_body"] or "(No popular answer)",
                "tags": ", ".join(record["tags"]) if record["tags"] else "",
            },
            metadata={
                "question_score": record["question_score"] or 0,
                "original_tags": record["tags"] or [],
            },
            id=record["id"],  # Use original ID for traceability
        )
    )

print(f"Prepared {len(records)} records")

In [ ]:
# Log records to Argilla (v2 uses log())
dataset.records.log(records)

print(f"Logged {len(records)} records to '{ARGILLA_DATASET_NAME}'")
print(f"\nOpen your Argilla Space to start curating!")

## Export Curated Data (Run After Annotation)

Once you've annotated records in Argilla, export them:

In [ ]:
# Load curated dataset from Argilla
dataset = client.datasets(name=ARGILLA_DATASET_NAME)

# Export to HuggingFace dataset format
hf_dataset = dataset.records.to_datasets()

print(f"Exported {len(hf_dataset)} records")
print(f"Columns: {hf_dataset.column_names}")

In [ ]:
# Filter to only records with responses (annotated)
# Then filter to "good" records based on your criteria

# Example: keep only records marked as high quality
# good_records = hf_dataset.filter(
#     lambda x: x.get("grammar_relevance") and "yes_high_quality" in str(x["grammar_relevance"])
# )

# Push to your HF Hub
# good_records.push_to_hub("your-name/jse-grammar-curated")

## SetFit: Train on labelled, predict the rest

Use your annotations to train a few-shot classifier, then push predictions back to Argilla as suggestions.

In [ ]:
!pip install -q setfit

In [ ]:
from setfit import SetFitModel, Trainer
from datasets import Dataset
from tqdm.auto import tqdm
from collections import Counter
import argilla as rg

# === SetFit Configuration ===
# Use ARGILLA_DATASET_NAME_FULL for full dataset workflow (recommended)
# Use ARGILLA_DATASET_NAME for filtered subset workflow
SETFIT_DATASET_NAME = ARGILLA_DATASET_NAME_FULL  # Change if using filtered workflow
QUESTION_NAME = "grammar_relevance"  # The question you've been labelling
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # Good for JP

# Minimum requirements for SetFit training
MIN_LABELLED_EXAMPLES = 8  # SetFit needs at least 8 examples per class
MIN_PER_CLASS = 2  # Absolute minimum per class

# === 1. Export records from Argilla via HF dataset ===
print(f"Fetching records from Argilla dataset '{SETFIT_DATASET_NAME}'...")

# Ensure client is initialised with HF_TOKEN for Space auth
client = rg.Argilla(
    api_url=ARGILLA_API_URL,
    api_key=ARGILLA_API_KEY,
    headers={"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else None,
)
dataset = client.datasets(name=SETFIT_DATASET_NAME)

# Export to HF dataset - this handles the API details
hf_ds = dataset.records.to_datasets()
print(f"Exported {len(hf_ds):,} records")
print(f"Columns: {hf_ds.column_names}")

# Check what the response column looks like
sample_responses = [r for r in hf_ds[QUESTION_NAME + ".responses"] if r][:3]
print(f"\nSample responses: {sample_responses}")

# Split into labelled/unlabelled
labelled = []
unlabelled = []

for i, record in enumerate(hf_ds):
    # Combine title + body as the text
    text = f"{record.get('title', '') or ''} {record.get('question_body', '') or ''}"
    record_id = record.get("id") or str(i)
    
    # Check for response - exported as "{question_name}.responses" column
    responses = record.get(f"{QUESTION_NAME}.responses")
    
    if responses and len(responses) > 0:
        # responses is a list of response values
        label = responses[0] if isinstance(responses[0], str) else str(responses[0])
        labelled.append({"text": text, "label": label, "record_id": record_id})
    else:
        unlabelled.append({"text": text, "record_id": record_id})

print(f"\nLabelled: {len(labelled):,}")
print(f"Unlabelled: {len(unlabelled):,}")

# Show label distribution
label_dist = Counter(r["label"] for r in labelled)
print(f"\nLabel distribution:")
for label, count in label_dist.most_common():
    print(f"  {label}: {count}")

# Validate we have enough data
if len(labelled) < MIN_LABELLED_EXAMPLES:
    print(f"\n⚠️  Only {len(labelled)} labelled examples found.")
    print(f"SetFit needs at least {MIN_LABELLED_EXAMPLES} examples total.")
    print("Label more records in Argilla before training.")
    raise SystemExit("Not enough labelled data")

min_class_size = min(label_dist.values()) if label_dist else 0
if min_class_size < MIN_PER_CLASS:
    print(f"\n⚠️  Imbalanced classes detected!")
    print(f"Smallest class has only {min_class_size} examples (need {MIN_PER_CLASS}+).")
    print("SetFit may struggle. Consider labelling more examples for underrepresented classes.")

print("\n✓ Data looks good for training!")

In [ ]:
# === 2. Train SetFit model ===

# Convert to HF Dataset
train_ds = Dataset.from_list([{"text": r["text"], "label": r["label"]} for r in labelled])

print(f"Training on {len(train_ds)} examples...")
print(f"Labels: {set(train_ds['label'])}")

# Sanity check: preview a few training examples
print("\n📝 Sample training texts:")
for i, ex in enumerate(labelled[:3]):
    preview = ex['text'][:300] + "..." if len(ex['text']) > 300 else ex['text']
    print(f"\n[{ex['label']}]")
    print(f"  {preview}")

# Load and train
model = SetFitModel.from_pretrained(MODEL_NAME)
trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    column_mapping={"text": "text", "label": "label"},
)

trainer.train()
print("\n✓ Training complete!")

In [ ]:
# === 3. Predict on unlabelled records ===

BATCH_SIZE = 256

print(f"Predicting {len(unlabelled):,} unlabelled records...")

# Predict in batches
predictions = []
texts = [r["text"] for r in unlabelled]

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting"):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_preds = model.predict(batch_texts)
    predictions.extend(batch_preds)

# Attach predictions to records
for record, pred in zip(unlabelled, predictions):
    record["prediction"] = pred

# Show prediction distribution
pred_dist = Counter(predictions)
print(f"\nPrediction distribution:")
for label, count in pred_dist.most_common():
    print(f"  {label}: {count}")

In [ ]:
# === 4. Push predictions to Argilla as suggestions ===
# Suggestions appear in the UI but don't count as responses until accepted
#
# ⚠️  IDEMPOTENCY NOTE: Running this cell multiple times will overwrite existing
# suggestions. This is safe (won't duplicate), but will replace any suggestions
# you may have manually edited in Argilla.

BATCH_SIZE = 500

print(f"About to push {len(unlabelled):,} suggestions to Argilla...")
print(f"Target dataset: '{SETFIT_DATASET_NAME}'")
print()

# Confirmation prompt (comment out if running non-interactively)
confirm = input("Continue? [y/N]: ").strip().lower()
if confirm != 'y':
    print("Aborted. No suggestions were pushed.")
    raise SystemExit("User cancelled")

print(f"\nPushing suggestions...")

# Build record updates with suggestions
dataset = client.datasets(name=SETFIT_DATASET_NAME)

for i in tqdm(range(0, len(unlabelled), BATCH_SIZE), desc="Uploading suggestions"):
    batch = unlabelled[i:i + BATCH_SIZE]
    
    updates = []
    for r in batch:
        updates.append(
            rg.Record(
                id=r["record_id"],
                suggestions=[
                    rg.Suggestion(
                        question_name=QUESTION_NAME,
                        value=r["prediction"],
                        agent="setfit-predictor",
                    )
                ],
            )
        )
    
    dataset.records.log(updates)

print(f"\n✓ Done! Suggestions added to '{SETFIT_DATASET_NAME}'")
print("Open Argilla to review — predictions appear as pre-filled suggestions.")

## Next Steps

1. **Curate in Argilla**: Open your HF Space and start labelling
2. **Export**: Run the export cells above when done
3. **SetFit**: Load your curated dataset for few-shot training:

```python
from setfit import SetFitModel, Trainer
from datasets import load_dataset

# Load your curated dataset
ds = load_dataset("your-name/jse-grammar-curated")

# Train SetFit model
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
trainer = Trainer(model=model, train_dataset=ds["train"])
trainer.train()
```

---

**Source manifest entry** (for `data/sets/sources.yaml`):

```yaml
japanese_stackexchange:
  hub: p1atdev/japanese-stackexchange
  subset: simple
  revision: <commit-hash>  # Pin this!
  filtered_to: your-name/jse-grammar-candidates
  curated_to: your-name/jse-grammar-curated
```

## HF Space Persistence (Optional)

Enable persistent storage for your Argilla HF Space so data survives restarts.

In [ ]:
# Enable persistent storage for your HF Space
# Run this once to enable storage — data will persist across restarts

from huggingface_hub import HfApi

# Extract Space repo ID from URL (e.g., "dylantonic/lingodingo-annotation")
# Update this to match your Space
SPACE_REPO_ID = "dylantonic/lingodingo-annotation"  # <-- UPDATE THIS

api = HfApi(token=HF_TOKEN)

# Request small storage (20GB free tier)
runtime = api.request_space_storage(repo_id=SPACE_REPO_ID, storage="small")
print(f"Storage requested: {runtime.storage}")

# Verify storage is enabled
runtime = api.get_space_runtime(SPACE_REPO_ID)
print(f"Current storage: {runtime.storage}")